# Week 12 ADV — Agentic Communication Workflows

Google Colab notebook สำหรับสร้างโปรเจกต์ `agentic-comm-automator` ตามโจทย์ Advanced Email & SMS

**ค่าเริ่มต้นเป็น DEMO MODE** จะไม่ส่ง Email/SMS จริงจนกว่าจะเปลี่ยน `DRY_RUN=false` และตั้งค่า `.env` ให้ครบ

In [ ]:
!pip -q install python-dotenv reportlab openpyxl
from pathlib import Path
import os, json, textwrap

PROJECT = Path('/content/agentic-comm-automator')
for folder in ['src', 'configs', 'data/reports', 'data/incoming_attachments', 'reports']:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
print('Project folders ready:', PROJECT)

## 1) สร้างไฟล์โปรแกรมทั้งหมด

In [ ]:
files = {}

files['src/__init__.py'] = '''"""Agentic communication automation package."""\n'''

files['src/utils.py'] = r'''import json, logging, os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
def setup_logging(name): return logging.getLogger(name)
def get_env_variable(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and not value: raise ValueError(f'Missing required environment variable: {name}')
    return value
def ensure_directory_exists(path): Path(path).mkdir(parents=True, exist_ok=True)
def log_audit_entry(logs, status, message, task_type='N/A', details=None):
    entry = {'timestamp': datetime.now().isoformat(timespec='seconds'), 'status': status,
             'task_type': task_type, 'message': message, 'details': details or {}}
    logs.append(entry)
    getattr(logging, 'error' if status in ('ERROR','CRITICAL') else 'info')(
        'AUDIT %s | %s | %s', status, task_type, message)
'''

files['src/template_renderer.py'] = r'''import string
class SafeDict(dict):
    def __missing__(self, key): return '{' + key + '}'
class TemplateRenderer:
    def render(self, template, context):
        # รองรับ {report_date}, {sales_figure:.2f} และ ${sales_figure:.2f}
        template = template.replace('${', '{')
        return string.Formatter().vformat(str(template), (), SafeDict(context))
'''

files['src/config_parser.py'] = r'''import json
from pathlib import Path
class ConfigParser:
    def __init__(self, config_path): self.config_path = Path(config_path)
    def load_config(self):
        if not self.config_path.exists(): raise FileNotFoundError(self.config_path)
        try: config = json.loads(self.config_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError as exc: raise ValueError(f'Invalid JSON: {exc}') from exc
        if not isinstance(config.get('tasks'), list): raise ValueError("Config must contain a 'tasks' list")
        for i, task in enumerate(config['tasks']):
            if not task.get('type') or not task.get('name'): raise ValueError(f'Task {i} needs type and name')
            if task['type'] not in {'send_email','send_sms','process_inbox'}: raise ValueError(f"Unsupported task type: {task['type']}")
        return config
'''

files['src/email_tasks.py'] = r'''import os, smtplib, imaplib, email
from email.message import EmailMessage
from email.header import decode_header
from pathlib import Path
from .utils import setup_logging, ensure_directory_exists
logger = setup_logging(__name__)
class EmailTasks:
    def __init__(self, smtp_server, smtp_port, imap_server, sender_email, sender_password, dry_run=True):
        self.smtp_server, self.smtp_port, self.imap_server = smtp_server, int(smtp_port), imap_server
        self.sender_email, self.sender_password, self.dry_run = sender_email, sender_password, dry_run
    def send_email(self, recipients, subject, body, attachments=None, is_html=False):
        recipients = recipients or {}; to = recipients.get('to', []); cc = recipients.get('cc', []); bcc = recipients.get('bcc', [])
        all_rcpts = list(to) + list(cc) + list(bcc)
        if not all_rcpts: logger.error('No recipients; email skipped'); return False
        if self.dry_run:
            logger.info('[DRY RUN] Email to=%s cc=%s bcc_count=%s subject=%s attachments=%s', to, cc, len(bcc), subject, attachments or [])
            return True
        msg = EmailMessage(); msg['From'] = self.sender_email; msg['To'] = ', '.join(to) if to else ''
        if cc: msg['Cc'] = ', '.join(cc)
        msg['Subject'] = subject
        if is_html: msg.set_content('HTML email; please view in an HTML-capable client.'); msg.add_alternative(body, subtype='html')
        else: msg.set_content(body)
        for item in attachments or []:
            path = Path(item)
            if not path.is_file(): logger.warning('Attachment missing, skipped: %s', path); continue
            data = path.read_bytes(); suffix = path.suffix.lower()
            subtype = 'pdf' if suffix == '.pdf' else ('vnd.openxmlformats-officedocument.spreadsheetml.sheet' if suffix == '.xlsx' else 'octet-stream')
            maintype = 'application'
            msg.add_attachment(data, maintype=maintype, subtype=subtype, filename=path.name)
        try:
            with smtplib.SMTP(self.smtp_server, self.smtp_port, timeout=30) as server:
                server.starttls(); server.login(self.sender_email, self.sender_password)
                server.send_message(msg, from_addr=self.sender_email, to_addrs=all_rcpts)
            logger.info('Email sent: %s', subject); return True
        except Exception as exc: logger.exception('Email send failed: %s', exc); return False
    def search_emails(self, mailbox='INBOX', criteria='UNSEEN', from_sender=None, subject_contains=None):
        if self.dry_run: logger.info('[DRY RUN] Inbox search simulated'); return []
        try:
            with imaplib.IMAP4_SSL(self.imap_server) as mail:
                mail.login(self.sender_email, self.sender_password); mail.select(mailbox)
                query = [criteria]
                if from_sender: query += ['FROM', f'"{from_sender}"']
                if subject_contains: query += ['SUBJECT', f'"{subject_contains}"']
                status, data = mail.search(None, *query)
                if status != 'OK': return []
                return data[0].split()
        except Exception as exc: logger.exception('Inbox search failed: %s', exc); return []
    @staticmethod
    def _decode(value):
        if not value: return ''
        parts = decode_header(value); out = ''
        for part, charset in parts:
            out += part.decode(charset or 'utf-8', errors='replace') if isinstance(part, bytes) else part
        return out
    def fetch_email_content(self, uid, mark_as_read=True, download_attachments_dir=None, pdf_only=False):
        if self.dry_run: return None
        try:
            with imaplib.IMAP4_SSL(self.imap_server) as mail:
                mail.login(self.sender_email, self.sender_password); mail.select('INBOX')
                status, data = mail.fetch(uid, '(RFC822)')
                if status != 'OK' or not data or not isinstance(data[0], tuple): return None
                msg = email.message_from_bytes(data[0][1]); text_body = ''; html_body = ''; saved = []
                for part in msg.walk():
                    if part.is_multipart(): continue
                    filename = part.get_filename(); disp = str(part.get('Content-Disposition', '')).lower()
                    if filename or 'attachment' in disp:
                        if not filename or not download_attachments_dir: continue
                        if pdf_only and not filename.lower().endswith('.pdf'): continue
                        folder = Path(download_attachments_dir); ensure_directory_exists(folder)
                        safe_name = Path(filename).name; target = folder / safe_name
                        target.write_bytes(part.get_payload(decode=True) or b''); saved.append({'filename': safe_name, 'filepath': str(target)})
                    elif part.get_content_type() == 'text/plain':
                        text_body += (part.get_payload(decode=True) or b'').decode(part.get_content_charset() or 'utf-8', errors='replace')
                    elif part.get_content_type() == 'text/html':
                        html_body += (part.get_payload(decode=True) or b'').decode(part.get_content_charset() or 'utf-8', errors='replace')
                if mark_as_read: mail.store(uid, '+FLAGS', '\\Seen')
                return {'uid': uid.decode() if isinstance(uid, bytes) else str(uid), 'sender': self._decode(msg.get('From')), 'subject': self._decode(msg.get('Subject')), 'text_body': text_body, 'html_body': html_body, 'has_attachments': bool(saved), 'attachments_meta': saved}
        except Exception as exc: logger.exception('Fetch email failed: %s', exc); return None
'''

files['src/sms_tasks.py'] = r'''from .utils import setup_logging
logger = setup_logging(__name__)
class SMSTasks:
    # Email-to-SMS gateways vary by country/carrier. Use an explicitly supplied gateway address when possible.
    def __init__(self, email_tasks): self.email_tasks = email_tasks
    def send_sms(self, phone_numbers, message, carrier_name=None):
        gateways = {'att':'@txt.att.net', 'verizon':'@vtext.com', 'tmobile':'@tmomail.net', 'sprint':'@messaging.sprintpcs.com', 'boost':'@sms.boostmobile.com'}
        results = []
        for number in phone_numbers:
            if '@' in number and '.' in number: address = number
            elif carrier_name and carrier_name.lower() in gateways: address = number + gateways[carrier_name.lower()]
            else: logger.warning('No supported SMS gateway configured for %s; skipped', number); results.append(False); continue
            results.append(self.email_tasks.send_email({'to':[address]}, '', message))
        return bool(results) and all(results)
'''

files['src/communication_agent.py'] = r'''import json, os, time
from pathlib import Path
from .utils import setup_logging, log_audit_entry, ensure_directory_exists, get_env_variable
from .email_tasks import EmailTasks
from .sms_tasks import SMSTasks
from .template_renderer import TemplateRenderer
logger = setup_logging(__name__)
class CommunicationAgent:
    def __init__(self, config, base_dir):
        self.config, self.base_dir = config, Path(base_dir); self.context = {}; self.audit_log = []
        self.renderer = TemplateRenderer(); dry_run = get_env_variable('DRY_RUN', 'true').lower() != 'false'
        self.email_tasks = EmailTasks(get_env_variable('SMTP_SERVER','smtp.gmail.com'), get_env_variable('SMTP_PORT','587'), get_env_variable('IMAP_SERVER','imap.gmail.com'), get_env_variable('SENDER_EMAIL',''), get_env_variable('SENDER_APP_PASSWORD',''), dry_run=dry_run)
        self.sms_tasks = SMSTasks(self.email_tasks)
    def render(self, value): return self.renderer.render(value, self.context)
    def process_task(self, task):
        typ, name, params = task['type'], task['name'], task.get('parameters', {})
        log_audit_entry(self.audit_log, 'INFO', f'Starting task: {name}', typ, {'task_name':name})
        try:
            if typ == 'send_email':
                rec = {kind:[self.render(x) for x in vals] for kind, vals in params.get('recipients', {}).items()}
                subject, body = self.render(params.get('subject_template','')), self.render(params.get('body_template',''))
                attachments = [str(self.base_dir / self.render(p)) for p in params.get('attachments', [])]
                ok = self.email_tasks.send_email(rec, subject, body, attachments, params.get('is_html', False))
                output = {'recipients':rec, 'subject':subject, 'attachments':attachments}
            elif typ == 'send_sms':
                nums = [self.render(x) for x in params.get('phone_numbers', [])]
                ok = self.sms_tasks.send_sms(nums, self.render(params.get('message_template','')), self.render(params.get('carrier_name','')))
                output = {'phone_numbers':nums}
            elif typ == 'process_inbox':
                search = params.get('search_criteria', {})
                uids = self.email_tasks.search_emails(params.get('mailbox','INBOX'), search.get('status','UNSEEN'), self.render(search['from_sender']) if search.get('from_sender') else None, self.render(search['subject_contains']) if search.get('subject_contains') else None)
                processed = []
                for uid in uids:
                    info = self.email_tasks.fetch_email_content(uid, True, self.base_dir / params.get('download_attachments_base_dir','data/incoming_attachments'))
                    if not info: continue
                    processed.append({'uid':info['uid'], 'subject':info['subject'], 'has_attachments':info['has_attachments']})
                    for rule in params.get('rules', []):
                        cond, actions = rule.get('if', {}), rule.get('then', {}); matched = True
                        if 'subject_contains' in cond and self.render(cond['subject_contains']).lower() not in info['subject'].lower(): matched = False
                        if 'body_contains' in cond and self.render(cond['body_contains']).lower() not in (info['text_body']+' '+info['html_body']).lower(): matched = False
                        if 'has_attachments' in cond and bool(cond['has_attachments']) != info['has_attachments']: matched = False
                        if not matched: continue
                        logger.info('Rule matched for email: %s', info['subject'])
                        merged = {**self.context, **info}
                        if actions.get('send_sms_alert'):
                            msg = self.renderer.render(actions.get('alert_message_template','Urgent email: {subject}'), merged)
                            nums = [self.render(x) for x in actions.get('alert_phone_numbers', [])]
                            self.sms_tasks.send_sms(nums, msg, self.render(actions.get('alert_carrier','')))
                        if 'reply_with_template' in actions:
                            reply = self.renderer.render(actions['reply_with_template'], merged)
                            subj = self.renderer.render(actions.get('reply_subject_template','Re: {subject}'), merged)
                            self.email_tasks.send_email({'to':[info['sender'].split('<')[-1].replace('>','').strip()]}, subj, reply)
                ok, output = True, {'processed_emails':processed}
            else: raise ValueError(f'Unknown task type: {typ}')
            key = task.get('output_context_key')
            if ok and key: self.context[key] = output
            log_audit_entry(self.audit_log, 'SUCCESS' if ok else 'ERROR', f'Task finished: {name}', typ, {'task_name':name})
        except Exception as exc:
            logger.exception('Task failed: %s', name)
            log_audit_entry(self.audit_log, 'ERROR', f'Task failed: {name}: {exc}', typ, {'error':str(exc)})
    def run(self):
        self.context.update({'report_name':'DailySalesSummary','report_date':__import__('datetime').date.today().isoformat(),'sales_figure':12345.67,'total_customers':500,'sender_email':get_env_variable('SENDER_EMAIL',''), 'test_recipient_email':get_env_variable('TEST_RECIPIENT_EMAIL','demo@example.com'), 'test_sms_phone_number':get_env_variable('TEST_SMS_PHONE_NUMBER','5550000000'), 'test_sms_carrier':get_env_variable('TEST_SMS_CARRIER','')})
        for task in self.config.get('tasks', []): self.process_task(task)
        report_dir = self.base_dir / 'reports'; ensure_directory_exists(report_dir)
        report_path = report_dir / 'audit_report.json'; report_path.write_text(json.dumps(self.audit_log, indent=2, ensure_ascii=False), encoding='utf-8')
        logger.info('Audit log saved to %s', report_path)
'''

files['main.py'] = r'''from pathlib import Path
from dotenv import load_dotenv
from src.config_parser import ConfigParser
from src.communication_agent import CommunicationAgent
from src.utils import ensure_directory_exists
BASE_DIR = Path(__file__).resolve().parent
def main():
    load_dotenv(BASE_DIR / '.env', override=True)
    for folder in ['data/reports','data/incoming_attachments','reports']: ensure_directory_exists(BASE_DIR / folder)
    config = ConfigParser(BASE_DIR / 'configs' / 'comm_pipeline_config.json').load_config()
    agent = CommunicationAgent(config, BASE_DIR); agent.run()
if __name__ == '__main__': main()
'''

files['configs/comm_pipeline_config.json'] = json.dumps({
  'tasks': [
    {'name':'Send Daily Sales Report Email','type':'send_email','parameters':{'recipients':{'to':['{test_recipient_email}'],'cc':['{sender_email}'],'bcc':[]},'subject_template':'Daily Sales Report - {report_date}','body_template':'Dear Team,\n\nSales date: {report_date}\nTotal sales: ${sales_figure:.2f}\nTotal customers: {total_customers}\n\nAutomated Agent','attachments':['data/reports/sales_report.pdf','data/reports/summary_data.xlsx'],'is_html':False},'output_context_key':'sales_email_result'},
    {'name':'Send Sales Alert SMS','type':'send_sms','parameters':{'phone_numbers':['{test_sms_phone_number}'],'message_template':'Sales {report_date}: ${sales_figure:.2f}; customers {total_customers}.','carrier_name':'{test_sms_carrier}'},'output_context_key':'sms_result'},
    {'name':'Simulate Incoming Request Email','type':'send_email','parameters':{'recipients':{'to':['{sender_email}']},'subject_template':'Data Request for {report_date} - Urgent','body_template':'Hello Agent, this is an urgent request for new data processing.','attachments':['data/reports/sales_report.pdf'],'is_html':False}},
    {'name':'Process Incoming Data Requests','type':'process_inbox','parameters':{'mailbox':'INBOX','search_criteria':{'status':'UNSEEN','from_sender':'{sender_email}','subject_contains':'Data Request'},'download_attachments_base_dir':'data/incoming_attachments','rules':[{'if':{'subject_contains':'Urgent','has_attachments':True},'then':{'send_sms_alert':True,'alert_phone_numbers':['{test_sms_phone_number}'],'alert_carrier':'{test_sms_carrier}','alert_message_template':'URGENT: request received from {sender}; subject: {subject}'}},{'if':{'body_contains':'new data processing'},'then':{'reply_with_template':'Agent received your request for {subject}. We are processing it now.','reply_subject_template':'Re: {subject} - Received & Processing'}}]},'output_context_key':'processed_inbox_results'}
  ]
}, indent=2, ensure_ascii=False)

files['.env'] = '''# DEMO SAFE DEFAULT: no real email/SMS is sent\nDRY_RUN=true\nSENDER_EMAIL=\nSENDER_APP_PASSWORD=\nSMTP_SERVER=smtp.gmail.com\nSMTP_PORT=587\nIMAP_SERVER=imap.gmail.com\nTEST_RECIPIENT_EMAIL=demo@example.com\nTEST_SMS_PHONE_NUMBER=5550000000\nTEST_SMS_CARRIER=\n'''
files['.gitignore'] = '''.env\n__pycache__/\n*.pyc\n*.log\ndata/incoming_attachments/\nreports/\n.ipynb_checkpoints/\n'''
files['README.md'] = '''# Week 12 ADV — Agentic Communication Workflows\n\nConfiguration-driven communication agent with dynamic templates, Email/SMS handlers, rule-based inbox processing and JSON audit logging.\n\n## Run\n1. Install dependencies: `pip install python-dotenv reportlab openpyxl`\n2. Configure `.env`; keep `DRY_RUN=true` for safe tests.\n3. Run: `python main.py`\n4. Audit output: `reports/audit_report.json`\n\nReal Gmail sending requires SMTP/IMAP access and a Gmail App Password. SMS email gateways are carrier-specific and may not work for Thai numbers. Never commit `.env`.\n'''

for relative, content in files.items():
    target = PROJECT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')

# Create the two sample attachments required by the assignment.
from reportlab.pdfgen import canvas
pdf_path = PROJECT / 'data/reports/sales_report.pdf'
c = canvas.Canvas(str(pdf_path)); c.drawString(72, 760, 'Sample Daily Sales Report'); c.drawString(72, 730, 'Sales: 12345.67'); c.drawString(72, 700, 'Customers: 500'); c.save()
from openpyxl import Workbook
wb = Workbook(); ws = wb.active; ws.title = 'Sales Summary'; ws.append(['Metric','Value']); ws.append(['Sales',12345.67]); ws.append(['Total Customers',500]); wb.save(PROJECT / 'data/reports/summary_data.xlsx')
print(f'Created {len(files)} project files and 2 sample attachments in {PROJECT}')

## 2) ตรวจสอบไฟล์และทดสอบแบบปลอดภัย (Dry Run)

In [ ]:
%cd /content/agentic-comm-automator
!python main.py

from pathlib import Path
print('\nAudit log exists:', Path('reports/audit_report.json').exists())
if Path('reports/audit_report.json').exists():
    print(Path('reports/audit_report.json').read_text(encoding='utf-8')[:2000])

## 3) ตั้งค่าบัญชี (ทำเฉพาะเมื่อต้องการทดสอบส่งจริง)

อย่าใส่รหัสผ่าน Gmail ปกติ ให้ใช้ **App Password** และอย่าแชร์หรือ commit ไฟล์ `.env` ไป GitHub สาธารณะ การส่ง SMS ผ่าน email gateway ขึ้นอยู่กับผู้ให้บริการและอาจใช้ไม่ได้กับเบอร์ไทย

In [ ]:
from getpass import getpass
from pathlib import Path
import os

sender = input('Gmail sender address: ').strip()
app_password = getpass('Gmail App Password (hidden): ').strip().replace(' ', '')
recipient = input('Test recipient email you control: ').strip()
phone = input('Phone number or full SMS gateway email (optional): ').strip()
carrier = input('Supported carrier key (att/verizon/tmobile/sprint/boost; blank if using gateway email): ').strip()
env_text = f'''DRY_RUN=true\nSENDER_EMAIL={sender}\nSENDER_APP_PASSWORD={app_password}\nSMTP_SERVER=smtp.gmail.com\nSMTP_PORT=587\nIMAP_SERVER=imap.gmail.com\nTEST_RECIPIENT_EMAIL={recipient}\nTEST_SMS_PHONE_NUMBER={phone or '5550000000'}\nTEST_SMS_CARRIER={carrier}\n'''
(PROJECT / '.env').write_text(env_text, encoding='utf-8')
print('Saved .env. DRY_RUN remains true; no real messages will be sent yet.')

## 4) เปิดการส่งจริง (ระวัง)

ตรวจสอบผู้รับและไฟล์แนบก่อนทุกครั้ง เพราะ workflow นี้จะส่งอีเมลรายงาน ส่งอีเมลจำลองคำขอเข้าหาตัวเอง และอาจตอบกลับ/ส่ง SMS ตามกฎที่กำหนด

In [ ]:
# รันเซลล์นี้เฉพาะเมื่อยืนยันว่าจะส่งข้อความจริงแล้วเท่านั้น
from pathlib import Path
env_path = PROJECT / '.env'
text_env = env_path.read_text(encoding='utf-8')
text_env = text_env.replace('DRY_RUN=true', 'DRY_RUN=false', 1)
env_path.write_text(text_env, encoding='utf-8')
print('DRY_RUN set to false. Run `!python main.py` in the project directory only after reviewing recipients/config.')

## 5) ดาวน์โหลดโปรเจกต์เป็น ZIP

ZIP นี้ไม่รวม `.env` เพื่อไม่ให้ข้อมูลลับติดไปด้วย

In [ ]:
import shutil
zip_base = '/content/agentic-comm-automator'
archive = shutil.make_archive(zip_base, 'zip', root_dir=PROJECT,
                              logger=None)
# Rebuild archive excluding .env for safer sharing
import zipfile
safe_zip = '/content/agentic-comm-automator-safe.zip'
with zipfile.ZipFile(safe_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in PROJECT.rglob('*'):
        if p.is_file() and p.name != '.env' and '__pycache__' not in p.parts:
            z.write(p, p.relative_to(PROJECT.parent))
from google.colab import files as colab_files
colab_files.download(safe_zip)